# Session 4 Solution: Execute a Cloud Evaluation Run

This notebook uses Entra authentication (`DefaultAzureCredential`) and the live Foundry eval API shape via `get_openai_client().evals` methods.

In [ ]:
from pathlib import Path
import sys
import time

candidates = [
    Path.cwd() / "solutions" / "src",
    Path.cwd().parent / "src",
    Path.cwd() / "src",
    Path.cwd().parent.parent / "solutions" / "src",
]
for path in candidates:
    if path.exists() and str(path) not in sys.path:
        sys.path.append(str(path))

from hackathon_solutions.config import load_config, validate_config
from hackathon_solutions.foundry_helpers import create_openai_client, load_jsonl

cfg = load_config()
validate_config(cfg)
openai_client = create_openai_client(cfg)
dataset_path = Path.cwd().parent / "data" / "session4_custom_eval_dataset.jsonl"
if not dataset_path.exists():
    dataset_path = Path.cwd() / "solutions" / "data" / "session4_custom_eval_dataset.jsonl"
rows = load_jsonl(dataset_path)
print("Rows loaded:", len(rows))

Rows loaded: 3


In [5]:
import os
import time
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import EvaluatorCategory, EvaluatorDefinitionType
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
)

# Azure AI Project endpoint
# Example: https://<account_name>.services.ai.azure.com/api/projects/<project_name>
endpoint = cfg.project_endpoint
# Model deployment name (required for prompt-based evaluators)
# Example: gpt-5-mini
model_deployment_name = cfg.model_deployment_name

# Create the project client
project_client = AIProjectClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(),
)

# Get the OpenAI client for evaluation API
client = project_client.get_openai_client()


code_evaluator = project_client.beta.evaluators.create_version(
    name="response_length_scorer",
    evaluator_version={
        "name": "response_length_scorer",
        "categories": [EvaluatorCategory.QUALITY],
        "display_name": "Response Length Scorer",
        "description": "Scores responses based on length, preferring 50-500 characters",
        "definition": {
            "type": EvaluatorDefinitionType.CODE,
            "code_text": (
                'def grade(sample: dict, item: dict) -> float:\n'
                '    """Score based on response length (prefer 50-500 chars)."""\n'
                '    response = item.get("response", "")\n'
                '    if not response:\n'
                '        return 0.0\n'
                '    length = len(response)\n'
                '    if length < 50:\n'
                '        return 0.2\n'
                '    elif length > 500:\n'
                '        return 0.5\n'
                '    return 1.0\n'
            ),
            "init_parameters": {
                "type": "object",
                "properties": {
                    "deployment_name": {"type": "string"},
                    "pass_threshold": {"type": "number"},
                },
                "required": ["deployment_name", "pass_threshold"],
            },
            "metrics": {
                "result": {
                    "type": "continuous",
                    "desirable_direction": "increase",
                    "min_value": 0.0,
                    "max_value": 1.0,
                }
            },
            "data_schema": {
                "type": "object",
                "required": ["item"],
                "properties": {
                    "item": {
                        "type": "object",
                        "properties": {
                            "response": {"type": "string"},
                        },
                    },
                },
            },
        },
    },
)

In [8]:
# Define the data schema
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "response": {"type": "string"},
        },
        "required": ["response"],
    },
)

# Reference both custom evaluators in testing criteria
testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "response_length_scorer",
        "evaluator_name": "response_length_scorer",
        "initialization_parameters": {
            "deployment_name": model_deployment_name,
            "pass_threshold": 0.5,
        },
    },
]

# Create the evaluation
eval_object = client.evals.create(
    name="custom-eval-test",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)

# Run the evaluation with inline data
# eval_run = client.evals.runs.create(
#     eval_id=eval_object.id,
#     name="custom-eval-run-01",
#     data_source=CreateEvalJSONLRunDataSourceParam(
#         type="jsonl",
#         source=SourceFileContent(
#             type="file_content",
#             content=[
#                 SourceFileContentContent(
#                     item={
#                         "response": "I'm sorry this watch isn't working for you. I'd be happy to help you with a replacement!",
#                     }
#                 ),
#                 SourceFileContentContent(
#                     item={
#                         "response": "I will not apologize for my behavior!",
#                     }
#                 ),
#             ],
#         ),
#     ),
# )

In [9]:
run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=f"{eval_object.id}-run",
    data_source={
        "type": "responses",
        "model": cfg.model_deployment_name,
        "source": {
            "type": "file_content",
            "content": [{"item": row} for row in rows]
        },
        "input_messages": {
            "type": "template",
            "template": [{"role": "user", "content": "{{item.query}}"}]
        },
        "sampling_params": {"temperature": 0.0}
    }
)
while str(run.status).lower() in {"queued", "in_progress"}:
    time.sleep(8)
    run = openai_client.evals.runs.retrieve(run_id=run.id, eval_id=eval_object.id)

{
    "run_id": run.id,
    "status": run.status,
    "report_url": run.report_url,
    "result_counts": run.result_counts.model_dump() if run.result_counts else None
}

{'run_id': 'evalrun_e6489c8f95274328985f13293bf42f78',
 'status': 'failed',
 'report_url': 'https://ai.azure.com/nextgen/r/TGmxZqOYSnWvA5cuV_bUiQ,SwedenAIML,,edwinfoundry0226,proj-default/build/evaluations/eval_ff63725846c142dfa01966c769b1846e/run/evalrun_e6489c8f95274328985f13293bf42f78',
 'result_counts': {'errored': 0, 'failed': 0, 'passed': 0, 'total': 0}}